# 04 Baseline Poisson Naïve Bayes Decoder (ThresholdCrossings, γ = 0)

This notebook implements the **baseline decoder** described in Issar et al. (2020),
*J Neurophysiol* 123:1472–1485.

## Context
The paper trains a neural network (NAS — Not A Sorter) that assigns each waveform
a spike-likelihood score Pspike ∈ [0,1].  A γ-threshold gates which waveforms are
used for decoding.  **γ = 0 means all threshold crossings are used** — this is the
baseline from which all NAS improvements are measured.

This notebook computes that baseline: a **Poisson Naïve Bayes decoder**
(paper §Materials and Methods, "Decoding") applied to the ThresholdCrossings
stream across all 312 sessions of Monkey N.

## Inputs
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/session_master_index.csv`
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/decoder_trial_table.csv`
- `/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory/candidate_stream_manifest.csv`
- Raw NWB files: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_baseline_decoder/session_baseline_accuracy.csv`
- `tables_baseline_decoder/decoder_unit_table.csv`
- `meta_baseline_decoder/baseline_decoder_report.json`
- `figures_baseline_decoder/fig01_*.{png,pdf}` … `fig07_*`

## Figures (paper-aligned)
| Figure | Paper analogue |
|---|---|
| fig01: Sample session accuracy report | Fig. 4A/B (γ = 0 baseline point) |
| fig02: Session-wise accuracy × trial count | Fig. 4A/B dual-axis |
| fig03: Accuracy vs days since first session | Fig. 5C |
| fig04: Early vs late session distributions | Fig. 5D |
| fig05: Session distribution histograms (4-panel) | Fig. 4C–F |
| fig06: Mean ± 1 SE normalised accuracy | Fig. 5E (γ = 0 slice) |
| fig07: Accuracy vs recording quality proxy | Fig. 5A/B |

In [ ]:
!pip install pynwb h5py

In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO